In [1]:
import bloqade.tsim
from bloqade import squin
import numpy as np
import stim
import tsim
import time

In [2]:
@squin.kernel
def main():
    q = squin.qalloc(3)
    squin.broadcast.reset(q)
    squin.h(q[0])
    squin.t(q[0])
    squin.h(q[0])
    squin.cx(q[0], q[2])
    for i in [0, 2]:
        # squin.depolarize(0.5, q[i])
        squin.cx(q[i], q[1])
    squin.measure(q[1])
    squin.cx(q[1], q[0])
    squin.broadcast.measure([q[0], q[2]])

In [3]:
c = bloqade.tsim.Circuit(main)
c.diagram(height=150)

In [4]:
sampler = c.compile_sampler()

In [5]:
samples = sampler.sample(shots=1_000_000)
samples

E0509 13:21:30.816130   24801 slow_operation_alarm.cc:73] Constant folding an instruction is taking > 1s:

  %gather = s32[1000000,1,4]{2,1,0} gather(%constant.89, %broadcast.256), offset_dims={1,2}, collapsed_slice_dims={}, start_index_map={0}, index_vector_dim=1, slice_sizes={1,4}, metadata={op_name="jit(evaluate)/gather" stack_frame_id=84}

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant folding from taking too long, but fundamentally you'll always be able to come up with an input program that takes a long time.

If you'd like to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
E0509 13:21:31.488504   24760 slow_operation_alarm.cc:140] The operation took 1.673092s
Constant folding an instruction is taking > 1s:

  %gather = s32[1000000,1,4]{2,1,0} gather(%constant.89, %broadcast.256), offset_dims={1,2}, collapsed_slice_dims={},

array([[False, False, False],
       [False,  True,  True],
       [False, False, False],
       ...,
       [False, False, False],
       [False, False, False],
       [False, False, False]], shape=(1000000, 3))

Tsim implements the approach described in

**Fast Classical Simulation of Quantum Circuits via Parametric Rewriting in the ZX-Calculus**

Matthew Sutcliffe and Aleks Kissinger (2024)

https://arxiv.org/abs/2403.06777

Circuits are represented as parameterized ZX diagrams, which are compiled into a contiguous datastructure for efficient sampling.

In [6]:
c.diagram("pyzx", scale=35);

Here, TSIM makes use of ZX diagrams in _doubled_ notation:


<img src="../figures/doubling.png" alt="doubled notation" width=650px>

In [7]:
tsim.Circuit("CNOT 0 1").diagram("pyzx");

In [8]:
tsim.Circuit("M 0").diagram("pyzx");

In [9]:
tsim.Circuit("""
    X 0
    R 0
""").diagram("pyzx");

In [10]:
tsim.Circuit("""
M !0
CNOT rec[-1] 1
""").diagram("pyzx");

Pauli Channels are represented as parameterized Pauli vertices:

In [11]:
tsim.Circuit("""
    Z_ERROR(0.01) 0
""").diagram("pyzx");

In [12]:
tsim.Circuit("""DEPOLARIZE2(0.01) 0 1""").diagram("pyzx");

For QEC experiments, we are often not interested in measurements themselves, but certain XOR combinations of them, referred to as _detectors_. We can directly represent detectors in ZX diagrams by applying classical XORs to measurement bits. The XOR has a simple ZX representation:

<img src="../figures/xor.png" alt="XOR gate" width=350px>

In [13]:
c = tsim.Circuit("""
    R 0 1
    H 0
    CNOT 0 1
    M 0 1
    DETECTOR rec[-1] rec[-2]
    OBSERVABLE_INCLUDE(0) rec[-1]
""")
c.diagram("pyzx");

In [14]:
sampler = c.compile_sampler()
sampler.sample(10)

array([[ True,  True],
       [ True,  True],
       [ True,  True],
       [False, False],
       [ True,  True],
       [False, False],
       [False, False],
       [False, False],
       [ True,  True],
       [ True,  True]])

In [15]:
detector_sampler = c.compile_detector_sampler()
detectors = detector_sampler.sample(10)

In [16]:
mask = detectors == 0

In [17]:
c = tsim.Circuit("""
    RX 0
    R 1 2 3 4 5
    T 0
    H 0
    CNOT 0 2
    CNOT 0 3
    CNOT 2 5
    TICK
    CX 0 1 3 4
    X_ERROR(0.01) 0 1 3 4
    TICK
    CX 2 1 5 4
    X_ERROR(0.01) 2 1 5 4
    TICK
    MR 1 4
    DETECTOR rec[-2]
    DETECTOR rec[-1]
    TICK
    CX 0 1 3 4
    X_ERROR(0.01) 0 1 3 4
    TICK
    CX 2 1 5 4
    X_ERROR(0.01) 2 1 5 4
    TICK
    MR 1 4
    DETECTOR rec[-2] rec[-4]
    DETECTOR rec[-1] rec[-3]
    M 0 2 3 5
    DETECTOR rec[-3] rec[-4] rec[-6]
    DETECTOR rec[-1] rec[-2] rec[-5]
    OBSERVABLE_INCLUDE(0) rec[-3]
    OBSERVABLE_INCLUDE(1) rec[-1]
""")

In [18]:
c.diagram(height=200)

In [19]:
c.diagram("pyzx", scale=30, scale_horizontally=1.4);

In [20]:
c.diagram("pyzx-dets", scale_horizontally=2);
# transform_error_basis=False


# Clifford sampling

In [21]:
p = 1e-4
stim_circuit = stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    rounds=7,
    distance=7,
    before_round_data_depolarization=p,
    before_measure_flip_probability=p,
    after_clifford_depolarization=p,
    after_reset_flip_probability=p,
)
stim_sampler = stim_circuit.compile_detector_sampler()

circuit = tsim.Circuit.from_stim_program(stim_circuit)
sampler = circuit.compile_detector_sampler()

In [22]:
circuit.without_noise().diagram("timeslice-svg", height=250, rows=1)

In [ ]:
circuit.diagram("pyzx", scale_horizontally=2)

Graph(10821 vertices, 11942 edges)

: 

In [ ]:
num_samples = 1024**2 * 64

start = time.perf_counter()
samples = sampler.sample(shots=num_samples)
end = time.perf_counter()
print(f"Time per shot (tsim): {(end - start) * 1e9 / num_samples:.1f} ns")

In [ ]:
num_samples = 1024
repeats = 1024 * 8

start = time.perf_counter()
for _ in range(repeats):
    samples = stim_sampler.sample(shots=num_samples)
end = time.perf_counter()
print(f"Time per shot (stim): {(end - start) * 1e9 / num_samples / repeats:.1f} ns")

In [ ]:
import sinter
import matplotlib.pyplot as plt

noise_vals = np.logspace(-3, -2, 4)
tasks = [
    sinter.Task(
        circuit=tsim.Circuit.from_stim_program(
            stim.Circuit.generated(
                "surface_code:rotated_memory_z",
                distance=distance,
                rounds=distance,
                after_clifford_depolarization=noise,
                before_round_data_depolarization=noise,
                before_measure_flip_probability=noise,
                after_reset_flip_probability=noise,
            )
        ).cast_to_stim(),
        json_metadata={"p": noise, "distance": distance, "rounds": 3},
    )
    for noise in noise_vals
    for distance in [3, 5, 7]
]

collected_stats = sinter.collect(
    num_workers=8,
    tasks=tasks,
    decoders=["pymatching"],
    max_shots=1024 * 256,
    start_batch_size=1024 * 256,
    max_batch_size=1024 * 256,
)

fig, ax = plt.subplots(1, 1)
sinter.plot_error_rate(
    ax=ax,
    stats=collected_stats,
    x_func=lambda stats: stats.json_metadata["p"],
    group_func=lambda stats: stats.json_metadata["distance"],
    failure_units_per_shot_func=lambda stats: stats.json_metadata["rounds"],
)
plt.plot(noise_vals, noise_vals, color="k", linestyle="--", lw=0.5, label="uncoded")
ax.loglog()
ax.set_xlabel("Physical Error Rate")
ax.set_ylabel("Logical Error Rate")
ax.legend();

## Non-Clifford circuits

In [ ]:
c = tsim.Circuit.from_file("../circuits/msd_5.stim")
c.diagram(height=500)

In [ ]:
c.diagram("pyzx-meas")

In [ ]:
sampler = c.compile_detector_sampler()

In [ ]:
num_samples = 50_000

start = time.perf_counter()
samples = sampler.sample(shots=num_samples)
end = time.perf_counter()
print(f"Time per shot (tsim): {(end - start) * 1e6 / num_samples:.2f} us")

On GPU, time per shot is about $0.6 \mu s$.

In [ ]:
sampler = c._stim_circ.compile_detector_sampler()

start = time.perf_counter()
num_samples = 1024
repeats = 1024
for i in range(repeats):
    samples = sampler.sample(shots=num_samples)
end = time.perf_counter()
print(f"Time per shot (stim): {(end - start) * 1e6 / num_samples / repeats:.2f} us")

In [ ]:
c = tsim.Circuit.from_file("../circuits/msc_3.stim")
print(c.tcount())
c.diagram("timeslice-svg", height=200, rows=1)

In [ ]:
c.diagram("pyzx-dets")

In [ ]:
sampler = c.compile_detector_sampler()

In [ ]:
sampler.sample(100)  # 6ms per shot

On GPU, Tsim achieves $100 \mu s$ per shot. However, the performance for this circuit has been improved by to $1\mu s$ on CPU using Tsim in **Simulating magic state cultivation with few Clifford terms**, Kwok Ho Wan & Zhenghao Zhong (2026) https://arxiv.org/pdf/2509.08658.